# Landslide Segmentation with YOLO11m-seg

This notebook is a cleaned and reproducible version of the original project notebook.

**Important:** the original notebook snapshot contains a Roboflow API key. This cleaned version does **not** store secrets. Enter the key at runtime or provide it through an environment variable.

The training configuration below preserves the main parameters used in the original notebook: YOLO11m-seg, 640 image size, batch size 16, AdamW, initial learning rate 1e-4, cosine learning-rate scheduling, weight decay 5e-4, horizontal/vertical flips, 90° rotation, and HSV augmentation.


## 1. Install dependencies

Run this cell in Google Colab or another GPU environment when the packages are not already installed.

In [ ]:
# Colab / notebook environment setup
!pip install -q ultralytics==8.4.33 roboflow pyyaml matplotlib opencv-python

## 2. Imports and reproducibility helpers

In [ ]:
import os
from pathlib import Path
from getpass import getpass

import torch
import yaml
from roboflow import Roboflow
from ultralytics import YOLO

print("Python environment ready.")
print("PyTorch version:", torch.__version__)
print("Ultralytics import: OK")

## 3. GPU check

The original run used a Tesla T4 GPU. The check below fails early if CUDA is unavailable.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available. Enable a GPU runtime before training.")

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"GPU: {gpu_name}")
print(f"GPU memory: {gpu_memory_gb:.2f} GB")

## 4. Roboflow configuration

The original notebook downloaded the dataset from Roboflow workspace `slas-workspace-ofcjv`, project `landslide-segmentation-zpy5e-kqanl`, version `1`. Those project identifiers are retained here, but the API key is never written into the notebook.

You can either set `ROBOFLOW_API_KEY` as an environment variable or enter it interactively when prompted.

In [ ]:
ROBOFLOW_WORKSPACE = "slas-workspace-ofcjv"
ROBOFLOW_PROJECT = "landslide-segmentation-zpy5e-kqanl"
ROBOFLOW_VERSION = 1

api_key = os.getenv("ROBOFLOW_API_KEY")
if not api_key:
    api_key = getpass("Enter your Roboflow API key (input hidden): ")

rf = Roboflow(api_key=api_key)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
dataset = version.download("yolov11")

DATASET_DIR = Path(dataset.location)
DATA_YAML = DATASET_DIR / "data.yaml"

print("Dataset location:", DATASET_DIR)
print("Dataset YAML:", DATA_YAML)

## 5. Inspect the dataset configuration

The original notebook created a second `landslide_data.yaml`, but the actual training call used Roboflow's `data.yaml`. This cleaned notebook uses the same YAML that the training process actually consumed. We do not hard-code the class count; it is read from the dataset configuration.

In [ ]:
with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

print(yaml.safe_dump(data_cfg, sort_keys=False))

print("Classes:", data_cfg.get("names"))
print("Number of classes:", data_cfg.get("nc", len(data_cfg.get("names", []))))

## 6. Verify dataset splits

This checks the expected `train`, `val`, and `test` image directories without creating unrelated empty folders.

In [ ]:
for split in ("train", "val", "test"):
    split_dir = DATASET_DIR / split / "images"
    exists = split_dir.exists()
    count = len(list(split_dir.iterdir())) if exists else 0
    print(f"{split:>5}: exists={exists}, files={count}, path={split_dir}")

## 7. Load YOLO11m-seg

The original project used the medium segmentation model `yolo11m-seg.pt`.

In [ ]:
MODEL_NAME = "yolo11m-seg.pt"
model = YOLO(MODEL_NAME)
print(f"Loaded model: {MODEL_NAME}")

## 8. Training configuration

These parameters preserve the main training configuration from the original notebook. The original code requested 100 epochs; the notebook snapshot does not by itself prove that all 100 epochs completed.

In [ ]:
TRAIN_CONFIG = {
    "data": str(DATA_YAML),
    "epochs": 100,
    "imgsz": 640,
    "batch": 16,
    "device": 0,
    "optimizer": "AdamW",
    "lr0": 0.0001,
    "cos_lr": True,
    "weight_decay": 0.0005,
    "fliplr": 0.5,
    "flipud": 0.5,
    "degrees": 90.0,
    "hsv_s": 0.5,
    "hsv_v": 0.4,
}

for key, value in TRAIN_CONFIG.items():
    print(f"{key}: {value}")

## 9. Train

Run this cell only when you are ready to spend GPU time. Ultralytics will save the training artifacts under the run directory.

In [ ]:
results = model.train(**TRAIN_CONFIG)
print("Training finished.")

## 10. Validate on the validation split

In [ ]:
print("Evaluating on validation split...")
val_metrics = model.val(data=str(DATA_YAML), split="val")

print(f"Validation box mAP50: {val_metrics.box.map50:.4f}")
print(f"Validation mask mAP50: {val_metrics.seg.map50:.4f}")
print(f"Validation box mAP50-95: {val_metrics.box.map:.4f}")
print(f"Validation mask mAP50-95: {val_metrics.seg.map:.4f}")

## 11. Evaluate on the test split

Use this only if the downloaded dataset contains a test split. Test metrics should be reported separately from validation metrics.

In [ ]:
test_images_dir = DATASET_DIR / "test" / "images"

if test_images_dir.exists() and any(test_images_dir.iterdir()):
    print("Evaluating on test split...")
    test_metrics = model.val(data=str(DATA_YAML), split="test")
    print(f"Test box mAP50: {test_metrics.box.map50:.4f}")
    print(f"Test mask mAP50: {test_metrics.seg.map50:.4f}")
    print(f"Test box mAP50-95: {test_metrics.box.map:.4f}")
    print(f"Test mask mAP50-95: {test_metrics.seg.map:.4f}")
else:
    print("No non-empty test image directory was found; skipping test evaluation.")

## 12. Run inference on test images

The original notebook called the prediction variable `test_image` but pointed it to the **training** image directory. This version correctly targets the test split when it exists.

In [ ]:
if test_images_dir.exists() and any(test_images_dir.iterdir()):
    prediction_results = model.predict(
        source=str(test_images_dir),
        conf=0.6,
        save=True,
        show_labels=True,
    )
    print("Prediction outputs were saved by Ultralytics.")
else:
    print("No test images available for inference.")

## 13. Save a lightweight run summary

This creates a machine-readable summary of the configuration used in the current run. It intentionally does not store secrets.

In [ ]:
import json

summary = {
    "model": MODEL_NAME,
    "dataset": {
        "workspace": ROBOFLOW_WORKSPACE,
        "project": ROBOFLOW_PROJECT,
        "version": ROBOFLOW_VERSION,
        "location": str(DATASET_DIR),
    },
    "training": TRAIN_CONFIG,
}

with open("run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved run_summary.json")